# Práctica 2 - Ejercicio 2

Asignatura: Programación para la Inteligencia Artificial

Alumno: Fernández Roldán, Daniel

Este es el segundo ejercicio de la Práctica 2 de Programación para la Inteligencia Artificial. Este ejercicio se debe entregar en un cuaderno de Jupyter separado del primero.

CIFAR100 es un conjunto de datos con dos niveles de etiquetado. Un etiquetado *fino* con 100 clases y un etiquetado *grueso* con 20 superclases. Se desea un sistema basado en neuronas lineales que tenga capacidad para realizar el etiquetado *fino*, pero se propone estudiar si la información que da el etiquetado *grueso* puede ayudar en la tarea aprovechando la probabilidad condicionada:

$$P(A|B) = \frac{P(A \cap B)}{P(B)}$$

Se espera una comparación entre la aproximación que usa solo un modelo neuronal para el etiquetado *fino* y la aproximación que incluye información del etiquetado *grueso*.

Información del conjunto de datos:

https://www.cs.toronto.edu/~kriz/cifar.html

Se espera el uso de las herramientas pertinentes tanto para completar el código como para realizar experimentos de los que se puedan extraer conclusiones sobre la capacidad del modelo entrenado. En el cuaderno se deben incluir los experimentos más relevantes.

Como conjunto de test se debe usar el conjunto de test íntegro que provee el conjunto de datos.

El cuaderno entregado debe llamarse ApellidosNombrePractica2Ejercicio2.ipynb

Consejos:
*  Cambiar el optimizador SGD por Adam. Lo veremos en clase próximamente :)
*  Normalizar los colores de las imágenes al rango [0,1].

In [1]:
# Import libraries.
import torch
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

import torch.nn as nn
import torch.optim as optim

In [2]:
# Define the transformation to convert images to tensors.
transform = transforms.ToTensor()

# Load the training dataset with the specified transformation.
train_dataset = datasets.CIFAR100(
    root = './data', train = True, download = True, transform = transform
)

# Load the test dataset with the same transformation.
test_dataset = datasets.CIFAR100(
    root = './data', train = False, download = True, transform = transform
)

# Create data loaders for the training and test datasets with a batch size of 256.
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

# Get a batch of images and labels from the training data loader.
images, fine_labels = next(iter(train_loader))

# Define the mapping from fine labels to coarse labels.
coarse_mapping = torch.tensor([
    4, 1, 14, 8, 0, 6, 7, 7, 18, 3, 3, 14, 9, 18, 7, 11, 3, 9, 7, 11,
    6, 11, 5, 10, 7, 6, 13, 15, 3, 15, 0, 11, 1, 10, 12, 14, 16, 9, 11, 5,
    5, 19, 8, 8, 15, 13, 14, 17, 18, 10, 16, 4, 17, 4, 2, 0, 17, 4, 18, 17,
    10, 3, 2, 12, 12, 16, 12, 1, 9, 19, 2, 10, 0, 1, 16, 12, 9, 13, 15, 13,
    16, 19, 2, 4, 6, 19, 5, 5, 8, 19, 18, 1, 2, 15, 6, 0, 17, 8, 14, 13
])

# Map the fine labels to coarse labels using the defined mapping. (Translate fine labels to coarse labels.)
coarse_labels = coarse_mapping[fine_labels]

# Print the dimensions of the images and labels.
print("Dimension of the images:", images.shape)
print("Dimension of the fine labels:", fine_labels.shape)
print("Dimension of the coarse labels:", coarse_labels.shape)   

Dimension of the images: torch.Size([256, 3, 32, 32])
Dimension of the fine labels: torch.Size([256])
Dimension of the coarse labels: torch.Size([256])


In [ ]:
# Define the CIFAR100 model architecture.
class CIFAR100_Model(nn.Module):
    def __init__(self):
        super(CIFAR100_Model, self).__init__()

        # Define the layers of the model.
        self.flatten = nn.Flatten()

        # Define the base layers of the model.
        self.base = nn.Sequential(
            nn.Linear(32 * 32 * 3, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU()
        )

        # Define the output layers for fine and coarse labels.
        self.fine_output = nn.Linear(256, 100)  # Output layer for fine labels (100 classes)

        # Define the output layer for coarse labels (20 classes).
        self.coarse_output = nn.Linear(256, 20)  # Output layer for coarse labels (20 classes)
    def forward(self, x):
        x = self.flatten(x)  # Flatten the input images.
        x= self.base(x)  # Pass through the base layers. In order to extract features from the images.

        coarse_prediction = self.coarse_output(x)  # Get coarse label predictions.
        fine_prediction = self.fine_output(x)  # Get fine label predictions.

        return coarse_prediction, fine_prediction  # Return both predictions.